<a href="https://colab.research.google.com/github/YazGonzalezHerrera/Simulaci-n-II/blob/main/Caminata%20sobre%20el%20grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def is_good_sequence(seq):
    """
    Verifica si una secuencia binaria (tupla o lista) es 'buena',
    es decir, no tiene unos (1s) adyacentes.
    """
    for i in range(len(seq) - 1):
        if seq[i] == 1 and seq[i + 1] == 1:
            return False
    return True

def get_good_sequences(m):
    """
    Genera todas las secuencias 'buenas' de longitud m.
    """
    good_seqs = []
    # Hay 2^m posibles secuencias binarias
    for i in range(1 << m):
        # Convertir el entero a una tupla de bits de longitud m
        seq = tuple((i >> (m - 1 - j)) & 1 for j in range(m))
        if is_good_sequence(seq):
            good_seqs.append(seq)
    return good_seqs

def step_random_walk(current_seq):
    """
    Realiza un paso de la caminata aleatoria a partir del estado actual.
    """
    m = len(current_seq)
    seq_list = list(current_seq)

    # Selecciona un componente al azar con probabilidad uniforme 1/m
    idx = np.random.randint(0, m)

    if seq_list[idx] == 1:
        # Si es 1, se cambia a 0 (siempre resulta en una secuencia buena)
        seq_list[idx] = 0
        return tuple(seq_list)
    else:
        # Si es 0, se intenta cambiar a 1
        seq_list[idx] = 1
        candidate_seq = tuple(seq_list)
        if is_good_sequence(candidate_seq):
            return candidate_seq
        else:
            # Si no es buena, se queda en el estado actual
            return current_seq

def build_transition_matrix(m):
    """
    Construye la matriz de transición P para las secuencias buenas de longitud m.
    """
    good_seqs = get_good_sequences(m)
    n = len(good_seqs)
    seq_to_idx = {seq: i for i, seq in enumerate(good_seqs)}

    P = np.zeros((n, n))

    for i, seq in enumerate(good_seqs):
        for bit_idx in range(m):
            seq_list = list(seq)
            if seq_list[bit_idx] == 1:
                seq_list[bit_idx] = 0
                next_seq = tuple(seq_list)
                j = seq_to_idx[next_seq]
                P[i, j] += 1.0 / m
            else:
                seq_list[bit_idx] = 1
                candidate_seq = tuple(seq_list)
                if is_good_sequence(candidate_seq):
                    j = seq_to_idx[candidate_seq]
                    P[i, j] += 1.0 / m
                else:
                    P[i, i] += 1.0 / m  # Permanece en el mismo estado

    return good_seqs, P

# --- Ejemplo de uso para m = 4 ---
m = 4
good_seqs, P = build_transition_matrix(m)

print(f"Secuencias buenas para m = {m} (Total: {len(good_seqs)}):")
for seq in good_seqs:
    print("".join(map(str, seq)))

print("\nMatriz de transición P:")
print(np.round(P, 3))

# Simulación de la caminata aleatoria
n_steps = 100000
current_state = (0, 0, 0, 0)
counts = {seq: 0 for seq in good_seqs}

for _ in range(n_steps):
    counts[current_state] += 1
    current_state = step_random_walk(current_state)

print("\nDistribución de frecuencias relativas tras la simulación:")
for seq, count in counts.items():
    print(f"Estado {''.join(map(str, seq))}: {count / n_steps:.4f}")

Secuencias buenas para m = 4 (Total: 8):
0000
0001
0010
0100
0101
1000
1001
1010

Matriz de transición P:
[[0.   0.25 0.25 0.25 0.   0.25 0.   0.  ]
 [0.25 0.25 0.   0.   0.25 0.   0.25 0.  ]
 [0.25 0.   0.5  0.   0.   0.   0.   0.25]
 [0.25 0.   0.   0.5  0.25 0.   0.   0.  ]
 [0.   0.25 0.   0.25 0.5  0.   0.   0.  ]
 [0.25 0.   0.   0.   0.   0.25 0.25 0.25]
 [0.   0.25 0.   0.   0.   0.25 0.5  0.  ]
 [0.   0.   0.25 0.   0.   0.25 0.   0.5 ]]

Distribución de frecuencias relativas tras la simulación:
Estado 0000: 0.1238
Estado 0001: 0.1245
Estado 0010: 0.1215
Estado 0100: 0.1272
Estado 0101: 0.1254
Estado 1000: 0.1259
Estado 1001: 0.1283
Estado 1010: 0.1234
